# Task 1: Dataset audit

### 1. Goal
| Задача / 任务 | Описание | Реализация | 任务描述 | 预计实施 |
|---|---|---|---|---|
| Ручной аудит датасета / 人工数据集审核 | Качество исходных данных напрямую определяет верхнюю границу качества модели. Необходимо системно проверить изображения и аннотации, выявить ошибки разметки и зафиксировать все внесённые изменения. | • Выявить неверные классы, пропущенные объекты и ложные аннотации.<br>• Найти дублирующиеся, слишком свободные и слишком тесные bounding boxes.<br>• Проверить выход координат за границы, пустые аннотации и несоответствие изображений и разметки.<br>• Проверить повреждённые изображения и отсутствие пар «изображение — аннотация».<br>• Не изменять исходную разметку без записи в журнале изменений. | 原始数据质量直接决定模型性能的上限。需要系统检查图像和标注，识别标注错误，并记录所有修改。 | • 检查类别错误、漏标目标和错误标注。<br>• 检查重复框、过松或过紧的边界框。<br>• 检查坐标越界、空标注以及图像与标注不匹配的问题。<br>• 检查损坏图像以及缺失的“图像—标注”文件对。<br>• 所有标注修改都必须记录在变更日志中，禁止无记录地直接修改原始标注。 |
### 2. Source dataset
### 3. Class conversion
### 4. Duplicate cleaning
### 5. Before and after comparison
### 6. Class distribution
### 7. Quality control
### 8. Limitations and Risks
### 9. A brief summary

### 2. Source dataset

In [2]:
from collections import Counter
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import display
from PIL import Image


# Find repository root.
current_path = Path.cwd().resolve()
repo_root = next(
    (
        path
        for path in (current_path, *current_path.parents)
        if (path / "data/raw").is_dir()
    ),
    None,
)

if repo_root is None:
    raise FileNotFoundError("Could not locate the ML_comp repository root")


image_dir = repo_root / "data/raw/images/train"
label_dir = repo_root / "data/raw/labels/train"
config_path = repo_root / "configs/baseline_v1/dataset.yaml"

IMAGE_SUFFIXES = {
    ".bmp",
    ".jpeg",
    ".jpg",
    ".png",
    ".tif",
    ".tiff",
    ".webp",
}

config = yaml.safe_load(config_path.read_text(encoding="utf-8"))

class_names = {
    int(class_id): str(class_name)
    for class_id, class_name in config["names"].items()
}


# Collect actual source files directly, without split manifests.
images = {
    path.stem: path
    for path in image_dir.iterdir()
    if (
        path.is_file()
        and not path.name.endswith(":Zone.Identifier")
        and path.suffix.lower() in IMAGE_SUFFIXES
    )
}

labels = {
    path.stem: path
    for path in label_dir.glob("*.txt")
    if (
        path.is_file()
        and not path.name.endswith(":Zone.Identifier")
    )
}


missing_labels = sorted(images.keys() - labels.keys())
missing_images = sorted(labels.keys() - images.keys())

empty_labels = []
corrupted_images = []
invalid_rows = []

object_counts = Counter()
image_counts = Counter()


# Validate images.
for stem, image_path in images.items():
    try:
        with Image.open(image_path) as image:
            image.verify()
    except Exception as exc:
        corrupted_images.append(
            {
                "stem": stem,
                "image_path": str(image_path),
                "error": f"{type(exc).__name__}: {exc}",
            }
        )


# Validate labels and calculate class statistics.
for stem, label_path in labels.items():
    lines = [
        line.strip()
        for line in label_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    if not lines:
        empty_labels.append(str(label_path))
        continue

    classes_in_image = set()

    for line_number, line in enumerate(lines, start=1):
        fields = line.split()

        if len(fields) != 5:
            invalid_rows.append(
                {
                    "stem": stem,
                    "line_number": line_number,
                    "reason": f"expected 5 fields, found {len(fields)}",
                    "content": line,
                }
            )
            continue

        try:
            class_id = int(fields[0])
            x_center, y_center, width, height = map(float, fields[1:])
        except ValueError:
            invalid_rows.append(
                {
                    "stem": stem,
                    "line_number": line_number,
                    "reason": "non-numeric value",
                    "content": line,
                }
            )
            continue

        if class_id not in class_names:
            invalid_rows.append(
                {
                    "stem": stem,
                    "line_number": line_number,
                    "reason": f"unknown class ID {class_id}",
                    "content": line,
                }
            )
            continue

        coordinates = (x_center, y_center, width, height)

        if not all(0 <= value <= 1 for value in coordinates):
            invalid_rows.append(
                {
                    "stem": stem,
                    "line_number": line_number,
                    "reason": "coordinates outside [0, 1]",
                    "content": line,
                }
            )
            continue

        if width <= 0 or height <= 0:
            invalid_rows.append(
                {
                    "stem": stem,
                    "line_number": line_number,
                    "reason": "non-positive box size",
                    "content": line,
                }
            )
            continue

        object_counts[class_id] += 1
        classes_in_image.add(class_id)

    for class_id in classes_in_image:
        image_counts[class_id] += 1


source_summary_df = pd.DataFrame(
    [
        {
            "images": len(images),
            "labels": len(labels),
            "objects": sum(object_counts.values()),
            "missing_labels": len(missing_labels),
            "missing_images": len(missing_images),
            "empty_labels": len(empty_labels),
            "corrupted_images": len(corrupted_images),
            "invalid_rows": len(invalid_rows),
        }
    ]
)

source_class_statistics_df = pd.DataFrame(
    [
        {
            "class_id": class_id,
            "class_name": class_name,
            "images": image_counts[class_id],
            "objects": object_counts[class_id],
            "object_share_percent": (
                object_counts[class_id] / sum(object_counts.values()) * 100
                if object_counts
                else 0
            ),
        }
        for class_id, class_name in class_names.items()
    ]
)


print("Source dataset summary")
display(source_summary_df)

print("Source dataset class distribution")
display(source_class_statistics_df)

if invalid_rows:
    print("Invalid annotation examples")
    display(pd.DataFrame(invalid_rows).head(20))
else:
    print("No structurally invalid annotation rows were found.")

Source dataset summary


,images,labels,objects,missing_labels,missing_images,empty_labels,corrupted_images,invalid_rows
0,4481,4481,20933,0,0,0,0,0


Source dataset class distribution


,class_id,class_name,images,objects,object_share_percent
0,0,HM,15,17,0.081211
1,1,LQS,25,30,0.143314
2,2,QHS,204,641,3.062151
3,3,MS,1188,1994,9.525629
4,4,A1_SU-35,278,1317,6.291501
5,5,A2_C-130,288,1297,6.195959
6,6,A3_C-17,257,998,4.767592
7,7,A4_C-5,116,500,2.388573
8,8,A5_F-16,240,1017,4.858358
9,9,A6_TU-160,86,361,1.724550


No structurally invalid annotation rows were found.


### Source dataset findings

The source dataset contains **4,481 images**, **4,481 annotation files**, and
**20,933 annotated objects** across 25 classes. Every image has a corresponding
annotation file. No missing files, empty annotations, corrupted images, or
structurally invalid YOLO rows were detected.

The dataset is strongly imbalanced. Aircraft account for approximately **85.3%**
of all objects, ships for **12.8%**, and the `FSC` class for **1.9%**.

The rarest classes are `HM` with 17 objects and `LQS` with 30 objects. The most
frequent class is `FA-18` with 2,147 objects. The ratio between the largest and
smallest classes is approximately **126:1**.

Therefore, the source dataset is technically complete and structurally valid,
but it has a severe class imbalance. Performance estimates for rare classes may
be less stable and more sensitive to individual samples. Structural validation
does not verify semantic annotation quality, so incorrect classes, missed
objects, duplicate boxes, and inaccurate box boundaries still require visual
inspection.